# 01 — EDA: volume geometry, class balance, and multi-site checks

Answers the 7 questions from `RESOURCES.md` / project discussion (2026-09-07),
grounded in the prior-art scan (step 0 of `structuring-ml-projects`):

1. Distribution of volume shape and voxel spacing across the 1362 training exams
2. Where the striatum falls and how large it is relative to the full volume
3. Class balance in `train_labels.csv`
4. Visual check of left-right asymmetry as a diagnostic signal (informs the
   flip-augmentation experiment, doesn't decide it up front)
5. (shape, spacing) clusters as a scanner proxy — check for confound with the target rate
6. NIfTI orientation/axis convention consistency
7. Background/intensity level, to calibrate a percentile-threshold background removal
   step (per Boulkrinat et al. 2025 / Martinez-Murcia et al.)

**Note on data handling:** cells that only read NIfTI header metadata (shape,
voxel spacing, affine/orientation) or aggregate label counts are separated from
cells that load actual pixel data (intensity stats, slice visualization) — the
latter are meant to be run and inspected by you, not by Claude, per the
project's AI-assistant data-handling rule (see `README.md`).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))
import config

%matplotlib inline

## 1. Volume geometry: shape and voxel spacing (header metadata only)

`nib.load()` is lazy — `.shape` and `.header.get_zooms()` read only the header,
not the pixel array, so this cell never touches actual scan intensities.

In [ ]:
def scan_geometry(nifti_dir: Path) -> pd.DataFrame:
    """Read shape and voxel spacing from every NIfTI header in nifti_dir.

    Header-only (lazy load) — does not read pixel data.
    """
    rows = []
    for path in sorted(nifti_dir.glob("*.nii.gz")):
        img = nib.load(str(path))
        zooms = img.header.get_zooms()
        rows.append(
            {
                "uid": path.name.removesuffix(".nii.gz"),
                "shape": img.shape,
                "spacing": tuple(round(float(z), 3) for z in zooms),
            }
        )
    return pd.DataFrame(rows)


geometry_df = scan_geometry(config.NIFTI_DIR)
print(f"{len(geometry_df)} volumes scanned")
geometry_df.head()

In [ ]:
shape_counts = geometry_df["shape"].value_counts()
spacing_counts = geometry_df["spacing"].value_counts()

print("Distinct shapes:", len(shape_counts))
print(shape_counts.head(20))
print()
print("Distinct spacings:", len(spacing_counts))
print(spacing_counts.head(20))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, axis_name in enumerate(["x", "y", "z"]):
    axes[i].hist(geometry_df["shape"].apply(lambda s: s[i]), bins=30)
    axes[i].set_title(f"Shape[{axis_name}]")
fig.suptitle("Volume dimension distribution")
plt.tight_layout()
plt.show()

**Decision this informs:** the resampling target for `data.py` — a single
target spacing (physical size preserved) vs. a single target shape (voxel
count preserved), per `deep-learning-imaging.md`. Pick based on how much
spread there actually is above.

## 2. Class balance (aggregate label counts only)

Reads `train_labels.csv` but only prints aggregate counts/percentages — no
per-row values are displayed.

In [ ]:
labels_df = pd.read_csv(config.TRAIN_LABELS_PATH)
class_counts = labels_df[config.TARGET_COLUMN].value_counts()
class_pct = labels_df[config.TARGET_COLUMN].value_counts(normalize=True) * 100

print("Class counts:\n", class_counts)
print("\nClass percentages:\n", class_pct.round(1))

class_counts.plot(kind="bar", title="is_pathologic class balance")
plt.show()

**Decision this informs:** whether `class_weight`/loss reweighting is
needed (Boulkrinat et al. 2025 used `class_weight="balanced"` on a roughly
balanced PPMI subset — check whether our imbalance, if any, is severe enough
to warrant the same) and the CV strategy (Stratified K-Fold regardless, but
confirms how important the stratification is).

## 3. (shape, spacing) clusters as a scanner proxy — confound check

Still metadata + aggregate label statistics only (group-level target rate,
not per-patient values).

In [ ]:
merged_df = geometry_df.merge(labels_df, on="uid", how="inner")
merged_df["scanner_proxy"] = list(zip(merged_df["shape"], merged_df["spacing"]))

proxy_summary = (
    merged_df.groupby("scanner_proxy")[config.TARGET_COLUMN]
    .agg(["count", "mean"])
    .rename(columns={"mean": "target_rate"})
    .sort_values("count", ascending=False)
)
print(f"{len(proxy_summary)} distinct (shape, spacing) groups")
proxy_summary.head(20)

**Decision this informs:** per `deep-learning-imaging.md`'s multi-site
generalization section — if `target_rate` varies sharply and systematically
across the larger groups above, that's a generalization risk to watch for
(and a reason to eventually check per-group validation performance), even
though this is a proxy, not a true site label.

## 4. NIfTI orientation/axis convention (header metadata only)

In [ ]:
def scan_orientation(nifti_dir: Path) -> pd.DataFrame:
    """Read the axis-code orientation (e.g. RAS, LAS) from every header."""
    rows = []
    for path in sorted(nifti_dir.glob("*.nii.gz")):
        img = nib.load(str(path))
        axcodes = nib.aff2axcodes(img.affine)
        rows.append({"uid": path.name.removesuffix(".nii.gz"), "orientation": "".join(axcodes)})
    return pd.DataFrame(rows)


orientation_df = scan_orientation(config.NIFTI_DIR)
print(orientation_df["orientation"].value_counts())

**Decision this informs:** whether a reorientation-to-canonical step
(`nib.as_closest_canonical` or similar) is needed in `data.py` before
resampling/cropping — if orientation is already consistent across all
volumes, this step is simpler; if not, it's mandatory before anything
spatial (per `deep-learning-imaging.md`, preprocessing step 3).

---
## The cells below load actual pixel data (intensities / images).

**Run these yourself and inspect the output** — per the project's
AI-assistant data-handling rule, Claude does not execute these cells or view
their image output.

## 5. Background/intensity level check

Following Boulkrinat et al. 2025: background removal thresholds at the
volume's 30th intensity percentile. Check whether that threshold choice
makes sense for our data by looking at its distribution across a sample of
volumes, and at overall intensity range/scale (relevant given multi-scanner
acquisition — see `deep-learning-imaging.md`'s normalization step).

In [ ]:
SAMPLE_SIZE = 50  # keep this cheap; it's a sanity check, not the final pipeline

sample_uids = geometry_df["uid"].sample(SAMPLE_SIZE, random_state=config.RANDOM_STATE)
intensity_rows = []
for uid in sample_uids:
    path = config.NIFTI_DIR / f"{uid}.nii.gz"
    data = nib.load(str(path)).get_fdata()
    intensity_rows.append(
        {
            "uid": uid,
            "min": data.min(),
            "max": data.max(),
            "p30": np.percentile(data, 30),
            "p99": np.percentile(data, 99),
        }
    )
intensity_df = pd.DataFrame(intensity_rows)
print(intensity_df.describe())

intensity_df[["p30", "p99"]].plot(kind="box", title="Intensity percentile spread across sampled volumes")
plt.show()

**Decision this informs:** the background-removal percentile and whether
per-volume normalization (not a single global constant) is required, given
multi-scanner intensity-scale differences.

## 6. Striatum location and left-right asymmetry (visual check)

Pick one normal and one abnormal example (by index, without printing which
uid is which class in any output you might share) and view central axial /
coronal / sagittal slices side by side. Look for:
- where the hot (striatal) region falls relative to the full volume, to size
  the centered crop in physical units;
- whether left-right asymmetry is visually apparent in the abnormal case —
  informs how to interpret the flip-augmentation experiment later, it
  doesn't decide it by itself (validate via the gate, per `RESOURCES.md`).

In [ ]:
example_uids = (
    labels_df.groupby(config.TARGET_COLUMN)[config.UID_COLUMN]
    .apply(lambda s: s.sample(1, random_state=config.RANDOM_STATE).iloc[0])
)

fig, axes = plt.subplots(len(example_uids), 3, figsize=(12, 4 * len(example_uids)))
for row, (label_value, uid) in enumerate(example_uids.items()):
    data = nib.load(str(config.NIFTI_DIR / f"{uid}.nii.gz")).get_fdata()
    cx, cy, cz = (s // 2 for s in data.shape)
    axes[row, 0].imshow(data[cx, :, :].T, cmap="hot", origin="lower")
    axes[row, 1].imshow(data[:, cy, :].T, cmap="hot", origin="lower")
    axes[row, 2].imshow(data[:, :, cz].T, cmap="hot", origin="lower")
    axes[row, 0].set_ylabel(f"class={label_value}")
for ax, title in zip(axes[0], ["sagittal", "coronal", "axial"]):
    ax.set_title(title)
plt.tight_layout()
plt.show()